动态规划求解最优策略需要事先知道环境的状态转移函数和奖励函数\
且只适用于有限的马尔可夫决策过程（状态空间和动作空间离散且有限）

In [5]:
import copy

In [1]:
class CliffWalkingEnv:

    def __init__(self, ncol=12, nrow=4):
        self.ncol = ncol  # 网格列数
        self.nrow = nrow  # 网格行数
        # 转移矩阵 P[state][action] = [(p, next_state, reward, done)]
        self.P = self.createP()

    def createP(self):
        P = [[[] for j in range(4)] for i in range(self.nrow * self.ncol)]
        change = [[0, -1], [0, 1], [-1, 0], [1, 0]]

        for i in range(self.nrow):
            for j in range(self.ncol):
                for a in range(4):
                    # 当前位置是悬崖 / 终点，任何动作都保持终止状态
                    if i == self.nrow - 1 and j > 0:
                        # 转移概率1，停留在自身，奖励0，回合结束
                        P[i * self.ncol + j][a] = [(1, i * self.ncol + j, 0, True)]
                        continue

                    # 计算移动后的坐标，边界截断，防止走出网格
                    next_x = min(self.ncol - 1, max(0, j + change[a][0]))
                    next_y = min(self.nrow - 1, max(0, i + change[a][1]))
                    next_state = next_y * self.ncol + next_x
                    reward = -1
                    done = False

                    # 判断下一个位置是不是悬崖 / 终点
                    if next_y == self.nrow - 1 and next_x > 0:
                        done = True
                        # 区分：终点（右下角）不是悬崖
                        if next_x != self.ncol - 1:
                            # 掉入悬崖
                            reward = -100

                    P[i * self.ncol + j][a] = [(1, next_state, reward, done)]
        return P

**策略评估**

由
$$
V^\pi(s) = \sum_{a\in\mathcal{A}} \pi(a \mid s) \left( \sum_{s'\in\mathcal{S}} P(s' \mid s,a) \left[ r(s,a,s') + \gamma  V^\pi(s') \right] \right)
$$
可得：
$$
V^{k+1}(s) = \sum_{a\in\mathcal{A}} \pi(a \mid s) \left( \sum_{s'\in\mathcal{S}} P(s' \mid s,a) \left[ r(s,a,s') + \gamma  V^{k+1}(s') \right] \right)
$$

所以在第0轮时，对每个状态赋任意初始值 $V^0(s)$\
通过动态规划不断更新下一轮的状态价值，$V^{k \to \infty} = V^\pi$

**策略提升**

在每个状态下，选取动作价值最大的动作，即：
$$
\pi'(s) = \arg\max_{a} Q^\pi(s,a) = \arg\max_{a}\left\{ r(s,a) + \gamma \sum_{s'} P(s' \mid s,a) V^\pi(s') \right\}
$$

**策略迭代**

1. 对当前策略进行评估，得到各个状态的状态价值函数
2. 根据状态价值函数，进行策略提升（贪心）
3. 重复步骤1、2，直到收敛到最优状态

In [2]:
class PolicyIteration:
    """策略迭代算法"""

    def __init__(self, env, theta, gamma):
        self.env = env
        # 状态价值V
        self.v = [0] * self.env.ncol * self.env.nrow
        # 策略pi[s][a]：状态s下选择动作a的概率，初始均匀随机 0.25
        self.pi = [[0.25, 0.25, 0.25, 0.25]
                   for i in range(self.env.ncol * self.env.nrow)]
        self.theta = theta    # 收敛阈值
        self.gamma = gamma    # 折扣因子

    def policy_evaluation(self):
        """策略评估"""

        cnt = 1
        while 1:
            max_diff = 0
            new_v = [0] * self.env.ncol * self.env.nrow
            for s in range(self.env.ncol * self.env.nrow):
                qsa_list = []
                for a in range(4):
                    qsa = 0
                    # 遍历环境所有转移分支
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                    # 加权：π(a|s) * Q(s,a)
                    qsa_list.append(self.pi[s][a] * qsa)
                new_v[s] = sum(qsa_list)
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))
            self.v = new_v
            if max_diff < self.theta:
                break
            cnt += 1
        print("策略评估进行%d轮后完成" % cnt)

    def policy_improvement(self):
        """策略提升"""
        
        for s in range(self.env.nrow * self.env.ncol):
            qsa_list = []
            for a in range(4):
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)
            maxq = max(qsa_list)
            # 统计有多少个动作同时达到最大Q值
            cntq = qsa_list.count(maxq)
            # 最优动作均分概率，其余动作概率0
            self.pi[s] = [1/cntq if q == maxq else 0 for q in qsa_list]
        print("策略提升完成")
        return self.pi

    def policy_iteration(self):
        """策略迭代"""
        
        while 1:
            self.policy_evaluation()
            old_pi = copy.deepcopy(self.pi)
            new_pi = self.policy_improvement()
            if old_pi == new_pi:
                break

In [12]:
def print_agent(agent, action_meaning, disaster=[], end=[]):
    """价值与策略可视化
    action_meaning: ['^','v','<','>'] 上下左右
    disaster：悬崖状态编号
    end：终点状态编号
    """
    print("状态价值：")
    for i in range(agent.env.nrow):
        for j in range(agent.env.ncol):
            print('%.3f'%agent.v[i * agent.env.ncol + j], end=' ')
        print()

    print("策略：")
    for i in range(agent.env.nrow):
        for j in range(agent.env.ncol):
            s = i * agent.env.ncol + j
            if s in disaster:
                print('****', end=' ')
            elif s in end:
                print('EEEEE', end=' ')
            else:
                a = agent.pi[s]
                pi_str = ''
                for k in range(len(action_meaning)):
                    pi_str += action_meaning[k] if a[k] > 0 else 'o'
                print(pi_str, end=' ')
        print()

In [13]:
env = CliffWalkingEnv()
action_meaning = ['^', 'v', '<', '>']
theta = 0.0001
gamma = 0.9
agent = PolicyIteration(env, theta, gamma)
agent.policy_iteration()
# 悬崖状态：37~46；终点47
print_agent(agent, action_meaning, list(range(37, 47)), [47])

策略评估进行75轮后完成
策略提升完成
策略评估进行94轮后完成
策略提升完成
策略评估进行64轮后完成
策略提升完成
策略评估进行12轮后完成
策略提升完成
策略评估进行1轮后完成
策略提升完成
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEEE 


**价值迭代**

由
$$
V^{*}(s) = \max_{a\in\mathcal{A}}\left\{ r(s,a) + \gamma \sum_{s'\in\mathcal{S}} P(s' \mid s,a) V^{*}(s') \right\}
$$
可得：
$$
V^{k+1}(s) = \max_{a\in\mathcal{A}}\left\{ r(s,a) + \gamma \sum_{s'\in\mathcal{S}} P(s' \mid s,a) V^{k}(s') \right\}
$$
根据该优化方程，利用动态规划求解每个状态下的最优价值\
在每一个状态下，选择价值函数最大的下一个状态，从而得到最优策略

In [14]:
class ValueIteration:
    """价值迭代算法"""

    def __init__(self, env, theta, gamma):
        self.env = env
        self.v = [0] * self.env.ncol * self.env.nrow
        self.theta = theta
        self.gamma = gamma
        # 保存最终收敛后的策略
        self.pi = [None for i in range(self.env.ncol * self.env.nrow)]

    def value_iteration(self):
        """价值迭代"""
        cnt = 0
        while 1:
            max_diff = 0  # 记录一轮迭代中价值最大变化量，用于判断收敛
            new_v = [0] * self.env.ncol * self.env.nrow
            # 遍历所有状态 s
            for s in range(self.env.ncol * self.env.nrow):
                qsa_list = []  # 存放当前状态s下所有动作a对应的Q(s,a)
                # 遍历全部4个动作：上、下、左、右
                for a in range(4):
                    qsa = 0
                    # 遍历转移概率模型 P(s'|s,a)
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        # 贝尔曼最优方程：Q(s,a) = Σ p [r + γ·V(s')·(1-done)]
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                    qsa_list.append(qsa)
                # V^{k+1}(s) = max_a Q(s,a)
                new_v[s] = max(qsa_list)
                # 更新最大价值差值
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))
            self.v = new_v
            # 收敛判定：所有状态价值变化都小于阈值theta
            if max_diff < self.theta:
                break
            cnt += 1
        print(f"价值迭代一共进行{cnt}轮")
        # 价值收敛后，提取最优策略
        self.get_policy()

    def get_policy(self):
        """根据收敛的最优价值函数，生成贪心最优策略"""

        for s in range(self.env.nrow * self.env.ncol):
            qsa_list = []
            for a in range(4):
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)
            maxq = max(qsa_list)
            cntq = qsa_list.count(maxq)  # 统计有多少动作达到最大Q值
            # 多重最优动作均分概率
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list]


In [15]:
env = CliffWalkingEnv()
action_meaning = ['^', 'v', '<', '>']  # 动作0~4对应：上、下、左、右
theta = 0.001    # 收敛阈值
gamma = 0.9      # 折扣系数
agent = ValueIteration(env, theta, gamma)
agent.value_iteration()
print_agent(agent, action_meaning, list(range(37, 47)), [47])

价值迭代一共进行14轮
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEEE 
